# CEI 220회 모델 상태 해시 독립 재계산 감사

이 노트북은 기존 학습을 다시 수행하지 않고, 로컬에 저장된 220개의
`model_state_only.pt`를 하나씩 CPU로 읽어 모델 상태 해시를 독립적으로
다시 계산합니다.

재계산한 해시는 다음 세 곳에 기록된 값과 비교합니다.

1. `aggregate_metrics.csv`
2. 각 run의 `metrics.json`
3. 각 run의 `hashes.json` 안의 `__model_state_dict__`

추가로 `model_state_only.pt` 파일 자체의 SHA-256도 다시 계산하여
`hashes.json`에 저장된 파일 해시와 비교합니다.

원본 모델 파일은 읽기만 하며 수정하지 않습니다.

In [ ]:
from pathlib import Path
import sys
import yaml

def find_project_root(start):
    start = Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "configs/experiment_220.yaml").is_file():
            return candidate
    raise FileNotFoundError("Open this notebook from inside the extracted CEI project.")

PROJECT_ROOT = find_project_root(Path.cwd())
CONFIG_PATH = PROJECT_ROOT / "configs/experiment_220.yaml"
config = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
RESULT_ROOT = (
    PROJECT_ROOT
    / config["project"]["output_root"]
    / config["project"]["experiment_name"]
)
SCRIPT = PROJECT_ROOT / "audit/audit_model_state_hashes.py"

print("Project root:", PROJECT_ROOT)
print("Full result folder:", RESULT_ROOT)
print("Audit script:", SCRIPT)
print("Python:", sys.executable)
assert SCRIPT.is_file(), "Audit script is missing."
assert (RESULT_ROOT / "aggregate_metrics.csv").is_file(), (
    "The distribution includes reference summaries only. Restore a working copy "
    "of the full result directory before running this audit."
)
print("Audit execution overwrites four audit report files in the working result folder.")


## 독립 재계산 실행

약 9.25GB의 모델 파일을 순차적으로 읽기 때문에 SSD 성능에 따라
대략 수 분에서 수십 분이 걸릴 수 있습니다. 한 번에 모델 하나만 CPU로
불러오므로 GPU 메모리는 사용하지 않으며 메모리 사용량도 제한적입니다.

In [ ]:
import subprocess

command = [
    sys.executable,
    str(SCRIPT),
    "--project-root",
    str(PROJECT_ROOT),
    "--config",
    str(CONFIG_PATH),
    "--result-root",
    str(RESULT_ROOT),
    "--expected-runs",
    "220",
    "--progress-every",
    "10",
]

print("실행 명령:", " ".join(command))
subprocess.run(command, cwd=PROJECT_ROOT, check=True)

## 결과 확인

모든 검사가 정상이라면 `all_passed`가 `true`이고,
`audit_failed_runs`가 0이어야 합니다.

In [ ]:
import json
import pandas as pd

summary = json.loads(
    (RESULT_ROOT / "model_state_rehash_summary.json").read_text(
        encoding="utf-8"
    )
)
audit = pd.read_csv(
    RESULT_ROOT / "model_state_rehash_audit.csv"
)
groups = pd.read_csv(
    RESULT_ROOT / "model_state_rehash_group_summary.csv"
)
mismatches = pd.read_csv(
    RESULT_ROOT / "model_state_rehash_mismatches.csv"
)

print(json.dumps(summary, ensure_ascii=False, indent=2))
display(groups)
print("불일치 run 수:", len(mismatches))
if len(mismatches):
    display(mismatches)

## 감사 결과의 보존

감사 성공·실패 여부와 관계없이 다음 네 파일을 함께 보존하고,
감사 결과를 공유할 때에도 함께 포함합니다.

- `model_state_rehash_summary.json`
- `model_state_rehash_audit.csv`
- `model_state_rehash_mismatches.csv`
- `model_state_rehash_group_summary.csv`

불일치가 없는 경우에도 불일치 결과 파일을 포함합니다. 재실행하면
이 네 파일을 덮어쓰므로, 기존 감사 결과는 별도로 보존하고
전체 결과 디렉터리의 작업 사본에서 감사를 수행합니다.